# SASRec — self-attentive sequence model on Yambda-50M

A causal-attention Transformer over each user's Listen+ history (a small GPT). Same GTS split / vocab as the baselines, so numbers are comparable.

**Bars to beat (our harness):** MostPop 0.0171, ItemKNN-bm25 **0.0709**. Paper SASRec (50M) = 0.0748.

**Settings:** *Accelerator* → **GPU** (T4 x2 or P100), *Internet* On. Optional `HF_TOKEN` secret.

In [ ]:
# Base install (torch is preinstalled on Kaggle GPU images).
!pip install -q --upgrade "git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
import os, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
import time, numpy as np
from ymrec.data.sequences import build_sequences

t0 = time.time()
data = build_sequences(size="50m", maxlen=200)
lens = np.array([len(s) for s in data.seqs])
print(f"built in {time.time()-t0:.1f}s | users={data.n_users:,} items={data.n_items:,} "
      f"eval_users={len(data.eval_pos):,} | seq len mean={lens.mean():.0f} median={int(np.median(lens))} max={lens.max()}")

In [ ]:
from ymrec.models.sasrec import train_and_eval

t0 = time.time()
model, best = train_and_eval(
    data,
    d=64, n_blocks=2, n_heads=1, dropout=0.2,
    epochs=120, batch_size=128, lr=1e-3, eval_every=10,
)
print(f"\ntrained in {(time.time()-t0)/60:.1f} min | best: {best}")

In [ ]:
import pandas as pd
ref = {"MostPop": 0.0171, "ItemKNN-bm25 (ours)": 0.0709,
       "SASRec (paper)": 0.0748, "SASRec (ours, best)": round(best["ndcg@10"], 4)}
pd.Series(ref, name="NDCG@10").sort_values(ascending=False).to_frame()